In [1]:
import os
import math
import json
import warnings
import itertools
from datetime import datetime
from typing import List, Tuple

warnings.filterwarnings("ignore")

In [2]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [1]:
import numpy as np 
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.utils.args import DefaultArgumentParser
from fintorch.utils.resample import resample_candles_df

In [4]:
string_args = [
    "--start-date", "2024-01-01",
    "--stop-date", "2024-06-01",
    "--time-frame", "60",
    "--symbol", "BTC-USDT"
]

args = DefaultArgumentParser.parse(args=string_args)

In [5]:
sdf = ONLINE_EXCHANGE.future.data.update_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame)
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=TimeFrame.MIN1).copy()

sdf

2024-04-07 00:36:00
2024-04-07 00:36:00
2024-04-07 00:40:00
2024-04-07 00:40:00


,open,high,low,close,volume,trade
timestamp,,,,,,
1705395660,43032.7,43040.0,42985.6,43003.7,201.772,3323
1705395720,43004.5,43016.0,42960.0,42967.3,534.556,4093
1705395780,42967.3,42976.5,42954.2,42957.8,117.148,1847
1705395840,42957.9,42965.7,42928.9,42942.2,245.105,3502
1705395900,42942.3,42953.0,42919.0,42942.6,134.988,1928
...,...,...,...,...,...,...
1712437560,68386.6,68391.8,68351.4,68351.5,48.246,1028
1712437620,68351.6,68405.0,68351.6,68405.0,46.587,876
1712437680,68405.0,68428.5,68404.9,68426.1,63.897,1154


In [3]:
df = sdf.copy()
destination_timeframe = TimeFrame.MIN1
source_timeframe = TimeFrame.MIN1
window_size = destination_timeframe // source_timeframe
first_timestamp = math.ceil(df.index[0] / destination_timeframe) * destination_timeframe

print(window_size)
print(first_timestamp)


df = df[first_timestamp <= df.index]
indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=window_size)

df['first'] = 0 == (df.index % destination_timeframe)
df['new-high'] = df.high.rolling(window=indexer, min_periods=1, step=window_size).max()
df['new-low'] = df.low.rolling(window=indexer, min_periods=1, step=window_size).min()
df['new-close'] = df.close.rolling(window=indexer, min_periods=1, step=window_size).agg(lambda items: list(items)[-1])
df['new-volume'] = df.volume.rolling(window=indexer, min_periods=1, step=window_size).sum()
# df = df.dropna()

df
# df[["first", "open", "high", "new-high", "low", "new-low", "close", "new-close", "volume", "new-volume"]].tail(20)

NameError: name 'sdf' is not defined

In [10]:
# timestamp = datetime.now().timestamp()
# ONLINE_EXCHANGE.future.data.prepare(symbols=[args.symbol], time_frames=[TimeFrame.MIN1, TimeFrame.DAY1])
# ONLINE_EXCHANGE.future.data.next(timestamp=timestamp)


ONLINE_EXCHANGE.future.data.update_candles_dataframe(symbol=args.symbol, time_frame=TimeFrame.MIN1)
df = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=TimeFrame.DAY1).copy()
df.insert(0, "datetime", [datetime.fromtimestamp(ts) for ts in df.index])

df

2024-04-07 00:41:00
2024-04-07 00:41:00
2024-04-07 00:41:00
2024-04-06 03:30:00


,datetime,open,high,low,close,volume,trade
timestamp,,,,,,,
1705449600,2024-01-17 03:30:00,43133.0,43198.0,42200.0,42783.4,209396.081,2896397.0
1705536000,2024-01-18 03:30:00,42783.3,42932.3,40673.6,41323.1,335080.756,4381862.0
1705622400,2024-01-19 03:30:00,41323.2,42197.0,40300.0,41662.2,345941.730,4485546.0
1705708800,2024-01-20 03:30:00,41662.1,41883.5,41453.6,41691.8,95866.062,1350118.0
1705795200,2024-01-21 03:30:00,41691.8,41880.9,41500.0,41580.0,67035.552,1011538.0
...,...,...,...,...,...,...,...
1712016000,2024-04-02 03:30:00,69700.0,69731.6,64603.2,65505.3,498973.144,6305719.0
1712102400,2024-04-03 03:30:00,65502.4,66940.0,64500.0,65971.9,274505.670,3708170.0
1712188800,2024-04-04 03:30:00,65971.8,69329.7,65100.0,68489.9,325562.882,4202676.0
